# Cloud Provider Analytics — exploración de fuentes

Primera entrega de 72.80 Big Data. Cargamos las 8 fuentes del dataset, las perfilamos con PySpark
y verificamos lo que afirmamos en el informe de diseño.

Garcia Olivia (61071) · Lombardo Abril (63655) · Morantes Agustín (61306)

## 1. Entorno

Instalamos PySpark y levantamos la sesión de Spark.

In [1]:
!pip install -q pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("tp-bigdata-exploracion") \
    .getOrCreate()

spark

## 2. Carga del dataset

Subimos el `.zip` del dataset y lo descomprimimos. Las fuentes quedan en `data/datalake/landing`.

In [3]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # elegí tu .zip acá

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('data')

DATA_DIR = 'data/datalake/landing'
print(os.listdir(DATA_DIR))

Saving cloud_provider_challenge_dataset_v1.zip to cloud_provider_challenge_dataset_v1.zip
['billing_monthly.csv', 'customers_orgs.csv', 'support_tickets.csv', 'nps_surveys.csv', 'marketing_touches.csv', 'resources.csv', 'usage_events_stream', 'users.csv']


### Fuentes batch

Leemos los 7 CSV y contamos las filas de cada uno.

In [4]:
archivos = {
    "customers_orgs": "customers_orgs.csv",
    "resources": "resources.csv",
    "users": "users.csv",
    "support_tickets": "support_tickets.csv",
    "nps_surveys": "nps_surveys.csv",
    "marketing_touches": "marketing_touches.csv",
    "billing_monthly": "billing_monthly.csv",
}

dfs = {}
for nombre, archivo in archivos.items():
    dfs[nombre] = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{DATA_DIR}/{archivo}")
    )
    print(f"{nombre}: {dfs[nombre].count()} filas")

customers_orgs: 80 filas
resources: 400 filas
users: 800 filas
support_tickets: 1000 filas
nps_surveys: 92 filas
marketing_touches: 1500 filas
billing_monthly: 240 filas


### Esquema y primeras filas

Miramos qué columnas tiene cada fuente, qué tipo detectó Spark y cómo son los datos reales.

In [5]:
for nombre, df in dfs.items():
    print(f"\n===== {nombre} =====")
    df.printSchema()
    df.show(5, truncate=False)


===== customers_orgs =====
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)

+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  |Education    |sa-east   |standard |tr

### Nulos por columna

Contamos los campos vacíos de cada columna. Es el insumo de la V de veracidad, y más adelante hay
que decidir columna por columna si el nulo es un estado válido o un dato que falta.

In [6]:
from pyspark.sql import functions as F

for nombre, df in dfs.items():
    print(f"\n===== {nombre}: nulos por columna =====")
    df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
    ]).show(truncate=False)


===== customers_orgs: nulos por columna =====
+------+--------+--------+---------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id|org_name|industry|hq_region|plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------+--------+--------+---------+---------+-------------+-----------+---------+---------------+----------------+---------+
|0     |0       |0       |0        |0        |0            |0          |0        |0              |0               |11       |
+------+--------+--------+---------+---------+-------------+-----------+---------+---------------+----------------+---------+


===== resources: nulos por columna =====
+-----------+------+-------+------+----------+-----+---------+
|resource_id|org_id|service|region|created_at|state|tags_json|
+-----------+------+-------+------+----------+-----+---------+
|0          |0     |0      |0     |0         |0    |83       |
+-----------+------+-------+

### Duplicados

Comparamos el total de filas contra las filas distintas de cada fuente.

In [7]:
for nombre, df in dfs.items():
    total = df.count()
    distinct = df.distinct().count()
    print(f"{nombre}: {total} filas, {distinct} distintas, {total - distinct} duplicadas")

customers_orgs: 80 filas, 80 distintas, 0 duplicadas
resources: 400 filas, 400 distintas, 0 duplicadas
users: 800 filas, 800 distintas, 0 duplicadas
support_tickets: 1000 filas, 1000 distintas, 0 duplicadas
nps_surveys: 92 filas, 92 distintas, 0 duplicadas
marketing_touches: 1500 filas, 1500 distintas, 0 duplicadas
billing_monthly: 240 filas, 240 distintas, 0 duplicadas


### Errores inyectados a propósito

El README del dataset avisa que hay problemas de calidad puestos a mano. Buscamos tres: facturas
con costo negativo, NPS fuera del rango válido y tickets resueltos antes de haber sido creados.

In [8]:
# 1) facturas con costo negativo (no debería pasar nunca)
print("Costos negativos en billing_monthly:")
dfs["billing_monthly"].filter(F.col("subtotal") < 0).show()

# 2) NPS fuera del rango válido (-100 a 100)
print("NPS fuera de rango en customers_orgs:")
dfs["customers_orgs"].filter(
    (F.col("nps_score") < -100) | (F.col("nps_score") > 100)
).show()

# 3) tickets "resueltos" antes de haber sido "creados" (imposible)
print("Fechas invertidas en support_tickets:")
dfs["support_tickets"].filter(
    F.col("resolved_at") < F.col("created_at")
).show()

Costos negativos en billing_monthly:
+------------+------------+----------+--------+-------+------+--------+--------------------+
|  invoice_id|      org_id|     month|subtotal|credits| taxes|currency|exchange_rate_to_usd|
+------------+------------+----------+--------+-------+------+--------+--------------------+
|inv_gx8uwtk7|org_g0fn9xuy|2025-08-01|-1125.54|   3.42|236.36|     USD|             1.00132|
|inv_bmzw7539|org_i7p5tb94|2025-08-01| -849.43|   18.2|178.38|     USD|             1.04212|
|inv_9rvn97y8|org_vyie6ivw|2025-08-01| -910.68|  70.98|191.24|     USD|             0.96732|
|inv_3tlo1i7x|org_x7eedtjv|2025-08-01| -879.97|  56.93|184.79|     USD|             0.95074|
|inv_4dqxin67|org_pac56t4u|2025-08-01| -701.47|   2.71|147.31|     USD|             0.96736|
|inv_0kg6umh4|org_ktakpuxq|2025-08-01|-1355.74|   NULL|284.71|     ARS|             0.00157|
|inv_gblckuhy|org_phr62gds|2025-06-01| -696.82|   2.12|146.33|     USD|             0.99772|
|inv_93pomv7m|org_phr62gds|2025-0

## 3. El stream de eventos

`usage_events_stream` son 120 archivos JSONL y es la única fuente con volumen de verdad, así que
la leemos aparte de los CSV.

In [9]:
usage_events = spark.read.json(f"{DATA_DIR}/usage_events_stream/*.jsonl")

print(f"Total de eventos: {usage_events.count()}")
usage_events.printSchema()
usage_events.show(5, truncate=False)

Total de eventos: 43200
root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)

+---------+------------------+----------------+------------+----------------+------------+----------+------------+--------------+---------+--------------------+--------+-------+
|carbon_kg|cost_usd_increment|event_id        |genai_tokens|metric          |org_id      |region    |resource_id |schema_version|service  |timestamp           |unit    |value  |
+---------+------------------+----------------+------------+----------------+----------

### Perfil del stream

Nulos por columna, cuántos eventos hay de cada versión de esquema, y si el campo `value` —que
viene como texto— trae algo que no sea un número.

In [10]:
from pyspark.sql import functions as F

# nulos por columna
usage_events.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in usage_events.columns
]).show(truncate=False)

# cuántos eventos hay de cada versión de esquema
usage_events.groupBy("schema_version").count().show()

# el campo "value" es string: ¿siempre contiene números, o hay basura mezclada?
usage_events.select("value").filter(
    ~F.col("value").rlike("^-?[0-9]+\\.?[0-9]*$")
).show(10, truncate=False)

+---------+------------------+--------+------------+------+------+------+-----------+--------------+-------+---------+----+-----+
|carbon_kg|cost_usd_increment|event_id|genai_tokens|metric|org_id|region|resource_id|schema_version|service|timestamp|unit|value|
+---------+------------------+--------+------------+------+------+------+-----------+--------------+-------+---------+----+-----+
|10800    |0                 |0       |40068       |0     |0     |0     |0          |0             |0      |0        |2075|877  |
+---------+------------------+--------+------------+------+------+------+-----------+--------------+-------+---------+----+-----+

+--------------+-----+
|schema_version|count|
+--------------+-----+
|             1|10800|
|             2|32400|
+--------------+-----+

+-----+
|value|
+-----+
+-----+



## 4. Flujo batch con lógica MapReduce

Calculamos el costo total por organización, servicio y día con la API de RDD en vez del DataFrame
API, para que se vea la correspondencia con las fases de MapReduce. `reduceByKey` además
pre-agrega dentro de cada partición antes del shuffle, que es lo que hace el combiner de
MapReduce clásico.

In [11]:
from pyspark.sql import functions as F

# separamos la fecha (sin hora) del timestamp del evento
eventos_con_fecha = usage_events.withColumn(
    "fecha", F.to_date("timestamp")
)

# MAP: convertimos a RDD de pares (clave, valor)
# clave = (org_id, service, fecha) | valor = cost_usd_increment
pares_mapeados = eventos_con_fecha.rdd.map(
    lambda fila: ((fila["org_id"], fila["service"], fila["fecha"]), fila["cost_usd_increment"] or 0.0)
)

# SHUFFLE + REDUCE: Spark agrupa por clave (shuffle) y reduceByKey
# suma los valores de cada clave (reduce)
costo_diario = pares_mapeados.reduceByKey(lambda a, b: a + b)

# ordenamos por costo descendente y mostramos los primeros 10
resultado = costo_diario.map(lambda kv: (kv[0][0], kv[0][1], kv[0][2], round(kv[1], 2)))
resultado_ordenado = resultado.sortBy(lambda x: -x[3])

for org, service, fecha, costo in resultado_ordenado.take(10):
    print(f"{org} | {service} | {fecha} | ${costo}")

print(f"\nTotal de combinaciones (org, servicio, día): {costo_diario.count()}")

org_rixa11dp | genai | 2025-08-24 | $340.53
org_c11ertj5 | genai | 2025-07-11 | $334.68
org_tvhhpbmy | genai | 2025-07-04 | $313.42
org_x7eedtjv | compute | 2025-07-09 | $311.25
org_5935a0l7 | compute | 2025-07-09 | $271.64
org_c11ertj5 | genai | 2025-08-05 | $265.24
org_5935a0l7 | compute | 2025-07-15 | $264.49
org_1t2tala7 | compute | 2025-08-23 | $257.36
org_pht0hl9x | analytics | 2025-08-25 | $257.04
org_kdgigatj | compute | 2025-07-17 | $251.64

Total de combinaciones (org, servicio, día): 11050


## 5. Verificaciones de calidad

Lo que sigue verifica las afirmaciones del informe en vez de darlas por buenas, y busca los
problemas que el pipeline va a tener que resolver.

### 5.1 Claves primarias

Chequeamos que las PK declaradas en el inventario de fuentes sean efectivamente únicas.
`nps_surveys` no tiene clave simple, así que probamos la compuesta.

In [12]:
claves = {
    "customers_orgs": "org_id",
    "resources": "resource_id",
    "users": "user_id",
    "support_tickets": "ticket_id",
    "marketing_touches": "touch_id",
    "billing_monthly": "invoice_id",
}

for nombre, pk in claves.items():
    total = dfs[nombre].count()
    distintas = dfs[nombre].select(pk).distinct().count()
    print(f"{nombre}: {total} filas, {distintas} {pk} distintos -> {'OK' if total == distintas else 'DUPLICADOS'}")

# nps_surveys no tiene PK simple, probamos la compuesta
total = dfs["nps_surveys"].count()
distintas = dfs["nps_surveys"].select("org_id", "survey_date").distinct().count()
print(f"nps_surveys: {total} filas, {distintas} (org_id, survey_date) distintos -> {'OK' if total == distintas else 'DUPLICADOS'}")

customers_orgs: 80 filas, 80 org_id distintos -> OK
resources: 400 filas, 400 resource_id distintos -> OK
users: 800 filas, 800 user_id distintos -> OK
support_tickets: 1000 filas, 1000 ticket_id distintos -> OK
marketing_touches: 1500 filas, 1500 touch_id distintos -> OK
billing_monthly: 240 filas, 240 invoice_id distintos -> OK
nps_surveys: 92 filas, 92 (org_id, survey_date) distintos -> OK


### 5.2 Integridad referencial

Dijimos que `org_id` conecta todas las tablas. Buscamos valores que aparezcan en las tablas hijas
pero no existan en `customers_orgs`, y lo mismo con `resource_id` en el stream.

In [13]:
orgs = dfs["customers_orgs"].select("org_id").distinct()

for nombre in ["resources", "users", "support_tickets", "nps_surveys", "marketing_touches", "billing_monthly"]:
    huerfanos = dfs[nombre].join(orgs, "org_id", "left_anti").count()
    print(f"{nombre}: {huerfanos} org_id que no existen en customers_orgs")

# el stream tambien referencia resources
recursos = dfs["resources"].select("resource_id").distinct()
print(f"usage_events: {usage_events.join(orgs, 'org_id', 'left_anti').count()} org_id huerfanos")
print(f"usage_events: {usage_events.join(recursos, 'resource_id', 'left_anti').count()} resource_id huerfanos")

resources: 0 org_id que no existen en customers_orgs
users: 0 org_id que no existen en customers_orgs
support_tickets: 0 org_id que no existen en customers_orgs
nps_surveys: 0 org_id que no existen en customers_orgs
marketing_touches: 0 org_id que no existen en customers_orgs
billing_monthly: 0 org_id que no existen en customers_orgs
usage_events: 0 org_id huerfanos
usage_events: 0 resource_id huerfanos


### 5.3 Subtotales negativos en billing

Hay 13 facturas con `subtotal` negativo: pueden ser notas de crédito o el signo dado vuelta. Si el
impuesto está calculado sobre el valor absoluto, lo único corrupto es el signo.

In [14]:
billing = dfs["billing_monthly"].withColumn(
    "ratio_taxes", F.col("taxes") / F.abs(F.col("subtotal"))
)

billing.groupBy((F.col("subtotal") < 0).alias("subtotal_negativo")).agg(
    F.count("*").alias("filas"),
    F.round(F.min("ratio_taxes"), 5).alias("ratio_min"),
    F.round(F.max("ratio_taxes"), 5).alias("ratio_max"),
).show()

+-----------------+-----+---------+---------+
|subtotal_negativo|filas|ratio_min|ratio_max|
+-----------------+-----+---------+---------+
|             true|   13|  0.20999|  0.21001|
|            false|  227|  0.20997|  0.21003|
+-----------------+-----+---------+---------+



### 5.4 Moneda mixta

Para poder sumar hay que pasar todo a USD, y una factura que ya está en USD tendría que tener
`exchange_rate_to_usd = 1`. Medimos también cuánto cambia el total si aplicamos la tasa en vez de
ignorarla: que ninguna sea 1 no significa que el error sea grande.

In [15]:
dfs["billing_monthly"].groupBy("currency").agg(
    F.count("*").alias("filas"),
    F.round(F.min("exchange_rate_to_usd"), 5).alias("rate_min"),
    F.round(F.max("exchange_rate_to_usd"), 5).alias("rate_max"),
    F.round(F.avg("exchange_rate_to_usd"), 5).alias("rate_medio"),
    F.round(F.stddev("exchange_rate_to_usd") / F.avg("exchange_rate_to_usd"), 3).alias("coef_var"),
).show()

usd_mal = dfs["billing_monthly"].filter(
    (F.col("currency") == "USD") & (F.col("exchange_rate_to_usd") != 1.0)
).count()
print(f"Facturas en USD con exchange_rate_to_usd distinto de 1: {usd_mal}")

#ojo: que ninguna sea 1 no significa que el error sea grande.
#medimos cuanto cambia el total si aplicamos el rate en vez de ignorarlo
usd = dfs["billing_monthly"].filter(F.col("currency") == "USD")
impacto = usd.select(
    F.sum(F.abs("subtotal")).alias("sin_rate"),
    F.sum(F.abs("subtotal") * F.col("exchange_rate_to_usd")).alias("con_rate"),
    F.round(F.max(F.abs(F.col("exchange_rate_to_usd") - 1)) * 100, 2).alias("desvio_max_pct"),
)
impacto.withColumn(
    "dif_agregada_pct", F.round((F.col("con_rate") / F.col("sin_rate") - 1) * 100, 2)
).show()

+--------+-----+--------+--------+----------+--------+
|currency|filas|rate_min|rate_max|rate_medio|coef_var|
+--------+-----+--------+--------+----------+--------+
|     ARS|   51| 0.00133| 0.00162|    0.0015|   0.043|
|     EUR|   29|  0.9981| 1.19808|   1.10165|   0.042|
|     USD|  160| 0.85463| 1.11791|   0.99769|   0.052|
+--------+-----+--------+--------+----------+--------+

Facturas en USD con exchange_rate_to_usd distinto de 1: 160
+------------------+--------------+--------------+----------------+
|          sin_rate|      con_rate|desvio_max_pct|dif_agregada_pct|
+------------------+--------------+--------------+----------------+
|124872.46999999996|124726.8606875|         14.54|           -0.12|
+------------------+--------------+--------------+----------------+



### 5.5 El NPS está en dos tablas

`customers_orgs.nps_score` y `nps_surveys` no pueden ser las dos la fuente de verdad. Comparamos
la última encuesta de cada organización contra el valor del maestro, y miramos cuál de las dos
está limpia.

In [16]:
from pyspark.sql.window import Window

ventana = Window.partitionBy("org_id").orderBy(F.col("survey_date").desc())

ultima = (
    dfs["nps_surveys"]
    .filter(F.col("nps_score").isNotNull())
    .withColumn("rn", F.row_number().over(ventana))
    .filter(F.col("rn") == 1)
    .select("org_id", F.col("nps_score").alias("nps_encuesta"))
)

comparacion = (
    dfs["customers_orgs"]
    .filter(F.col("nps_score").isNotNull())
    .select("org_id", F.col("nps_score").alias("nps_maestro"))
    .join(ultima, "org_id")
)

print(f"orgs con valor en las dos fuentes: {comparacion.count()}")
print(f"orgs donde los dos valores coinciden: {comparacion.filter(F.col('nps_maestro') == F.col('nps_encuesta')).count()}")
comparacion.show(5)

# y vemos cual de las dos fuentes esta limpia
dfs["customers_orgs"].select(F.min("nps_score").alias("min"), F.max("nps_score").alias("max")).show()
dfs["nps_surveys"].select(F.min("nps_score").alias("min"), F.max("nps_score").alias("max")).show()

orgs con valor en las dos fuentes: 44
orgs donde los dos valores coinciden: 0
+------------+-----------+------------+
|      org_id|nps_maestro|nps_encuesta|
+------------+-----------+------------+
|org_1swjckjl|       -4.0|        43.0|
|org_253j2d54|       14.0|        35.0|
|org_3t60rjiw|       41.0|        34.0|
|org_53lc58dr|       11.0|        12.0|
|org_5935a0l7|        2.0|        20.0|
+------------+-----------+------------+
only showing top 5 rows
+-----+-----+
|  min|  max|
+-----+-----+
|-38.0|101.0|
+-----+-----+

+-----+----+
|  min| max|
+-----+----+
|-16.0|68.0|
+-----+----+



### 5.6 Evolución de esquema

Confirmamos que `carbon_kg` nulo sea exactamente el esquema v1 y no otra cosa. Con `genai_tokens`
no alcanza con ser v2: además tiene que ser un evento del servicio `genai`.

In [17]:
usage_events.groupBy("schema_version").agg(
    F.count("*").alias("eventos"),
    F.sum(F.col("carbon_kg").isNull().cast("int")).alias("carbon_kg_nulos"),
    F.sum(F.col("genai_tokens").isNull().cast("int")).alias("genai_tokens_nulos"),
).orderBy("schema_version").show()

# con genai_tokens no alcanza con ser v2, ademas tiene que ser servicio genai
usage_events.groupBy("schema_version", (F.col("service") == "genai").alias("es_genai")).agg(
    F.count("*").alias("eventos"),
    F.sum(F.col("genai_tokens").isNull().cast("int")).alias("genai_tokens_nulos"),
).orderBy("schema_version", "es_genai").show()

+--------------+-------+---------------+------------------+
|schema_version|eventos|carbon_kg_nulos|genai_tokens_nulos|
+--------------+-------+---------------+------------------+
|             1|  10800|          10800|             10800|
|             2|  32400|              0|             29268|
+--------------+-------+---------------+------------------+

+--------------+--------+-------+------------------+
|schema_version|es_genai|eventos|genai_tokens_nulos|
+--------------+--------+-------+------------------+
|             1|   false|   9774|              9774|
|             1|    true|   1026|              1026|
|             2|   false|  29268|             29268|
|             2|    true|   3132|                 0|
+--------------+--------+-------+------------------+



### 5.7 Rango temporal y clave de deduplicación

Buscamos dónde cae exactamente el corte de esquema. Fijamos la zona horaria en UTC porque los
timestamps vienen en UTC: sin eso Spark usa la de la máquina y el corte aparece corrido unas
horas. Después vemos si `event_id`, que va a ser la clave de dedupe del streaming, ya trae
duplicados.

In [18]:
#los timestamps del stream vienen en UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

eventos_ts = usage_events.withColumn("ts", F.to_timestamp("timestamp"))

eventos_ts.select(
    F.min("ts").alias("primer_evento"),
    F.max("ts").alias("ultimo_evento"),
    F.datediff(F.max("ts"), F.min("ts")).alias("dias"),
).show(truncate=False)

eventos_ts.groupBy("schema_version").agg(
    F.min("ts").alias("desde"),
    F.max("ts").alias("hasta"),
).orderBy("schema_version").show(truncate=False)

# event_id va a ser la clave de deduplicacion del streaming, vemos si ya viene con duplicados
total = usage_events.count()
distintos = usage_events.select("event_id").distinct().count()
print(f"usage_events: {total} eventos, {distintos} event_id distintos, {total - distintos} duplicados")

+-------------------+-------------------+----+
|primer_evento      |ultimo_evento      |dias|
+-------------------+-------------------+----+
|2025-07-03 00:02:00|2025-08-31 23:58:00|59  |
+-------------------+-------------------+----+

+--------------+-------------------+-------------------+
|schema_version|desde              |hasta              |
+--------------+-------------------+-------------------+
|1             |2025-07-03 00:02:00|2025-07-17 23:56:00|
|2             |2025-07-18 00:01:00|2025-08-31 23:58:00|
+--------------+-------------------+-------------------+

usage_events: 43200 eventos, 43200 event_id distintos, 0 duplicados


### 5.8 Costos negativos y spikes

La consigna avisa que los hay. Los contamos y miramos la distribución del costo, que es lo que
justifica detectar anomalías por percentiles en vez de promediar.

In [19]:
negativos = usage_events.filter(F.col("cost_usd_increment") < 0).count()
print(f"Eventos con cost_usd_increment negativo: {negativos}")

usage_events.select(
    F.expr("percentile_approx(cost_usd_increment, 0.5)").alias("p50"),
    F.expr("percentile_approx(cost_usd_increment, 0.99)").alias("p99"),
    F.max("cost_usd_increment").alias("maximo"),
).show()

# que metricas trae el stream: de aca salen las features del mart
usage_events.groupBy("metric", "unit").count().orderBy("metric").show()

Eventos con cost_usd_increment negativo: 216
+------+-------+--------+
|   p50|    p99|  maximo|
+------+-------+--------+
|1.0043|16.6934|317.4308|
+------+-------+--------+

+----------------+--------+-----+
|          metric|    unit|count|
+----------------+--------+-----+
|       cpu_hours|    NULL|  505|
|       cpu_hours|   hours|10187|
|        requests|    NULL|  916|
|        requests|   count|18596|
|storage_gb_hours|    NULL|  654|
|storage_gb_hours|gb_hours|12342|
+----------------+--------+-----+



### 5.9 Vocabularios

¿Hay que mapear regiones y servicios entre tablas, o ya usan los mismos valores?

In [20]:
print("Regiones en resources:")
dfs["resources"].select("region").distinct().orderBy("region").show(truncate=False)

print("Regiones en customers_orgs:")
dfs["customers_orgs"].select("hq_region").distinct().orderBy("hq_region").show(truncate=False)

print("Servicios en resources:")
dfs["resources"].select("service").distinct().orderBy("service").show(truncate=False)

print("Servicios en usage_events:")
usage_events.select("service").distinct().orderBy("service").show(truncate=False)

Regiones en resources:
+------------+
|region      |
+------------+
|ap-northeast|
|ap-south    |
|eu-central  |
|eu-west     |
|sa-east     |
|us-east     |
|us-west     |
+------------+

Regiones en customers_orgs:
+------------+
|hq_region   |
+------------+
|ap-northeast|
|ap-south    |
|eu-central  |
|eu-west     |
|sa-east     |
|us-east     |
|us-west     |
+------------+

Servicios en resources:
+----------+
|service   |
+----------+
|analytics |
|compute   |
|database  |
|genai     |
|networking|
|storage   |
+----------+

Servicios en usage_events:
+----------+
|service   |
+----------+
|analytics |
|compute   |
|database  |
|genai     |
|networking|
|storage   |
+----------+



### 5.10 Escala de CSAT

No está documentada en ningún lado, así que la deducimos de los datos.

In [21]:
dfs["support_tickets"].groupBy("csat").count().orderBy("csat").show()

+----+-----+
|csat|count|
+----+-----+
|NULL|  254|
| 0.0|   11|
| 1.0|   45|
| 2.0|  127|
| 3.0|  242|
| 4.0|  197|
| 5.0|   95|
| 6.0|   28|
| 7.0|    1|
+----+-----+

